# Few-Shot Learning & In-Context Learning
>
Fornece **exemplos rotulados diretamente no prompt** para que o modelo aprenda o padrão esperado sem fine-tuning. Brown et al. (2020) estabeleceram que GPT-3 exibia forte generalização a partir de poucos exemplos — fundamento empírico do In-Context Learning. A qualidade depende da seleção e ordem dos exemplos: rótulos consistentes, diversidade de casos e formato rigoroso são mais importantes que a quantidade.

**Referência:** Brown et al. (2020) *Language Models are Few-Shot Learners.* arXiv:2005.14165

In [ ]:
!pip install -q --upgrade langchain-ollama langchain-core python-dotenv requests


In [1]:
from pathlib import Path
import os, json, re, unicodedata
import requests
from textwrap import dedent

from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama

# ── Configuração local Ollama ──────────────────────────────────────────────
load_dotenv(override=True)

MODEL_NAME = "gemma3:4b"
CREATIVE_MODEL_NAME = "phi4-mini"
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434")

# ── LLM clients ────────────────────────────────────────────────────────────
llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=0,
)
llm_criativo = ChatOllama(
    model=CREATIVE_MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=0.7,
)

print(f"✓ modelo padrão: {MODEL_NAME}")
print(f"✓ modelo criativo: {CREATIVE_MODEL_NAME}")
print(f"✓ Ollama: {OLLAMA_BASE_URL}")

try:
    version = requests.get(f"{OLLAMA_BASE_URL}/api/version", timeout=3).json().get("version")
    print(f"✓ Ollama respondeu: {version}")
except Exception as exc:
    print(f"⚠ Ollama não respondeu em {OLLAMA_BASE_URL}: {exc}")
    print("  Se estiver usando kernel WSL, configure OLLAMA_BASE_URL para o endereço acessível do Ollama no Windows.")

print("Configuração concluída.")

# ── Funções auxiliares ────────────────────────────────────────────────────────
def normalizar(texto: str) -> str:
    """Lowercase + strip diacritics for keyword matching."""
    texto = unicodedata.normalize("NFKD", texto.lower())
    return "".join(ch for ch in texto if not unicodedata.combining(ch))

def extrair_json(texto: str) -> dict:
    """Strip markdown fences and parse the first JSON object found."""
    texto = texto.strip()
    if texto.startswith("```"):
        texto = re.sub(r"^```(?:json)?\s*|\s*```$", "", texto, flags=re.S).strip()
    inicio = texto.find("{")
    fim    = texto.rfind("}")
    if inicio != -1 and fim != -1 and fim > inicio:
        texto = texto[inicio : fim + 1]
    return json.loads(texto)

def chamar_texto(llm_client, prompt_template, alternativa: str, **kwargs) -> str:
    """Call the LLM and return a string; fall back gracefully."""
    if llm_client is None:
        return alternativa
    try:
        bruto = (prompt_template | llm_client).invoke(kwargs)
        return getattr(bruto, "content", str(bruto)).strip()
    except Exception as exc:
        print(f"⚠ Ollama: {exc}. Usando alternativa.")
        return alternativa

def pedir_json(llm_client, prompt_template, alternativa: dict, **kwargs) -> dict:
    """Call the LLM and parse JSON; fall back gracefully."""
    if llm_client is None:
        return alternativa
    try:
        bruto = (prompt_template | llm_client).invoke(kwargs)
        return extrair_json(getattr(bruto, "content", str(bruto)))
    except Exception as exc:
        print(f"⚠ Ollama: {exc}. Usando alternativa.")
        return alternativa


✓ modelo padrão: gemma3:4b
✓ modelo criativo: phi4-mini
✓ Ollama: http://172.18.224.1:11434
✓ Ollama respondeu: 0.23.2
Configuração concluída.


## 01. Few-Shot: Classificação de Sentimento

Três exemplos no prompt ensinam o modelo o formato e os rótulos esperados — sem fine-tuning.

In [2]:
# ── Few-Shot: Classificação de Sentimento ─────────────────────────────────
# Cenário: time de CX quer rotular reviews de produto automaticamente.
# 3 exemplos no prompt ensinam o modelo o formato e os rótulos esperados.

EXEMPLOS_SENTIMENTO = (
    "Texto: O atendimento foi impecável e a entrega chegou antes do prazo!\n"
    "Sentimento: Positivo\n\n"
    "Texto: O aplicativo trava toda vez que tento finalizar a compra.\n"
    "Sentimento: Negativo\n\n"
    "Texto: O produto chegou na data informada, embalagem íntegra.\n"
    "Sentimento: Neutro\n\n"
)

def classificar_sentimento(texto_entrada: str) -> str:
    """Classifica texto do cliente em Positivo / Negativo / Neutro via few-shot."""
    prompt = PromptTemplate(
        input_variables=["exemplos", "texto_entrada"],
        template=(
            "Classifique o sentimento do texto como Positivo, Negativo ou Neutro.\n\n"
            "Exemplos:\n{exemplos}"
            "Texto: {texto_entrada}\n"
            "Sentimento:"
        ),
    )
    resposta = chamar_texto(
        llm,
        prompt,
        alternativa="Positivo",
        exemplos=EXEMPLOS_SENTIMENTO,
        texto_entrada=texto_entrada,
    )
    # Mantem apenas a primeira linha e remove qualquer rótulo repetido
    return resposta.splitlines()[0].replace("Sentimento:", "").strip()


TEXTOS_TESTE = [
    "Estou simplesmente apaixonado por este novo produto — superou minhas expectativas!",
    "Péssimo. O item veio errado e o suporte não resolveu em três tentativas.",
    "Recebi o pedido conforme descrito no site.",
    "GenAi muda o twempo todo, mas tudo é opensource!"
]

for texto in TEXTOS_TESTE:
    print(f"Texto:      {texto}")
    print(f"Sentimento: {classificar_sentimento(texto)}\n")


Texto:      Estou simplesmente apaixonado por este novo produto — superou minhas expectativas!
Sentimento: Positivo

Texto:      Péssimo. O item veio errado e o suporte não resolveu em três tentativas.
Sentimento: Negativo

Texto:      Recebi o pedido conforme descrito no site.
Sentimento: Neutro

Texto:      GenAi muda o twempo todo, mas tudo é opensource!
Sentimento: Neutro



## 02. In-Context Learning: Transformação de Padrão

O modelo aprende uma transformação não-óbvia apenas observando exemplos entrada→saída.

In [5]:
# ── In-Context Learning: aprendizado de padrão via exemplos ───────────────
# O modelo aprende uma transformação não-óbvia (inversão de string)
# apenas com 2 exemplos no prompt — sem nenhuma instrução adicional.

def aplicar_transformacao_icl(
    descricao_tarefa: str,
    exemplos: list[dict],
    texto_entrada: str,
) -> str:
    """Aplica um padrão aprendido com exemplos em contexto."""
    # Monta o bloco few-shot dinamicamente
    bloco_exemplos = "".join(
        f"Entrada: {e['entrada']}\nSaída: {e['saida']}\n\n"
        for e in exemplos
    )
    prompt = PromptTemplate(
        input_variables=["descricao_tarefa", "exemplos", "texto_entrada"],
        template=(
            "Tarefa: {descricao_tarefa}\n\n"
            "Exemplos:\n{exemplos}"
            "Entrada: {texto_entrada}\n"
            "Saída:"
        ),
    )
    return chamar_texto(
        llm,
        prompt,
        alternativa=texto_entrada[::-1],   # alternativa simples: inversão com Python
        descricao_tarefa=descricao_tarefa,
        exemplos=bloco_exemplos,
        texto_entrada=texto_entrada,
    )


DESCRICAO = "Inverta as letras de cada palavra, mantendo a posição das palavras."
EXEMPLOS_ICL = [
    {"entrada": "sol", "saida": "los"},
    {"entrada": "casa", "saida": "asac"},
    {"entrada": "Bom Dia", "saida": "aiD moB"},
]

for teste in ["python", "machine learning", "dados", "A aula termina logo"]:
    resultado = aplicar_transformacao_icl(DESCRICAO, EXEMPLOS_ICL, teste)
    print(f"  Entrada: {teste:<20}  Saída: {resultado}")


  Entrada: python                Saída: python
Saída: nohtyp
  Entrada: machine learning      Saída: neihc lem gnilearuc
  Entrada: dados                 Saída: Saída: sadoD
  Entrada: A aula termina logo   Saída: A alua netirmais logos


## 03. Avaliação

Meça a acurácia do classificador few-shot em um conjunto de casos de teste.

In [6]:
# ── Avaliação: acurácia do classificador few-shot ─────────────────────────

def avaliar_classificador(funcao_modelo, casos_teste: list[dict]) -> float:
    """Executa o classificador em todos os casos de teste e imprime um relatório."""
    acertos = 0
    print(f"{'Texto':<50} {'Esperado':<10} {'Previsto':<10} OK")
    print("-" * 80)
    for caso in casos_teste:
        predicao = funcao_modelo(caso["entrada"]).strip()
        correto  = predicao.lower() == caso["rotulo"].lower()
        acertos += int(correto)
        flag = "✓" if correto else "✗"
        print(f"{caso['entrada'][:48]:<50} {caso['rotulo']:<10} {predicao:<10} {flag}")
    acuracia = acertos / len(casos_teste)
    print(f"\nAcurácia: {acuracia:.0%}  ({acertos}/{len(casos_teste)})")
    return acuracia


CASOS_TESTE = [
    {"entrada": "Este curso de IA é simplesmente fantástico!",         "rotulo": "Positivo"},
    {"entrada": "O suporte demorou horas para responder ao meu caso.", "rotulo": "Negativo"},
    {"entrada": "O pacote foi entregue na portaria do prédio.",        "rotulo": "Neutro"},
    {"entrada": "Adorei! Voltarei a comprar com certeza.",             "rotulo": "Positivo"},
    {"entrada": "Produto veio danificado e sem nota fiscal.",          "rotulo": "Negativo"},

    {"entrada": "Estudar estatística não é facil.",          "rotulo": "Positivo"},

]

avaliar_classificador(classificar_sentimento, CASOS_TESTE)


Texto                                              Esperado   Previsto   OK
--------------------------------------------------------------------------------
Este curso de IA é simplesmente fantástico!        Positivo   Positivo   ✓
O suporte demorou horas para responder ao meu ca   Negativo   Negativo   ✓
O pacote foi entregue na portaria do prédio.       Neutro     Neutro     ✓
Adorei! Voltarei a comprar com certeza.            Positivo   Positivo   ✓
Produto veio danificado e sem nota fiscal.         Negativo   Negativo   ✓
Estudar estatística não é facil.                   Positivo   Negativo   ✗

Acurácia: 83%  (5/6)


0.8333333333333334

In [7]:
llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=0,
)


In [8]:
## Exemplo local usando Ollama, sem chave de API e sem custo -------------------

import requests
import json

url = f"{OLLAMA_BASE_URL}/api/chat"

payload = {
    "model": MODEL_NAME,
    "messages": [
        {"role": "system", "content": "Você é um especialista em Inteligência Artificial Generativa."},
        {"role": "user", "content": "Explique o que são LLMs e como ganhar dinheiro com eles."},
    ],
    "stream": False,
    "options": {
        "temperature": 0.7,
        "num_predict": 500,
    },
}

response = requests.post(url, json=payload, timeout=120)
response.raise_for_status()
resposta = response.json()

print("=== Status da Resposta ===")
print(f"Status Code       : {response.status_code}")
print(f"Sucesso?          : {response.ok}")

print("\n=== Modelo ===")
print(resposta.get("model"))

print("\n=== Resposta ===")
print(resposta["message"]["content"])

## Comparando com o segundo modelo local
payload["model"] = CREATIVE_MODEL_NAME
response_2 = requests.post(url, json=payload, timeout=120)
response_2.raise_for_status()
resposta_2 = response_2.json()

print("\n=== Resposta com segundo modelo ===")
print(f"Modelo: {resposta_2.get('model')}")
print(resposta_2["message"]["content"])

## FIM -------------------------------------------------------------------------


=== Status da Resposta ===
Status Code       : 200
Sucesso?          : True

=== Modelo ===
gemma3:4b

=== Resposta ===
Com prazer! Como especialista em Inteligência Artificial Generativa, posso te explicar tudo sobre LLMs (Large Language Models - Modelos de Linguagem Grandes) e como você pode começar a lucrar com eles.

**O que são LLMs?**

LLMs são um tipo de modelo de IA que foram treinados em quantidades massivas de dados de texto da internet. Pense em livros, artigos, sites, código – praticamente tudo que é escrito digitalmente. Essa exposição massiva permite que eles aprendam padrões complexos na linguagem, como:

*   **Compreensão da Linguagem:** Eles conseguem entender o significado de palavras, frases e textos, mesmo em diferentes contextos.
*   **Geração de Texto:** Eles podem gerar texto novo, coerente e relevante, imitando o estilo e a estrutura da linguagem que foram treinados.
*   **Tarefas de Linguagem:** Eles podem realizar uma variedade de tarefas relacionadas à lingua